In [ ]:
!pip install timm torch torchvision scikit-learn matplotlib seaborn open-clip-torch
!pip install h5py

In [ ]:
# ============================================================================
# STEP 2: Import Libraries
# ============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
import timm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================================
# STEP 3: Set Random Seeds for Reproducibility
# ============================================================================
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
# ============================================================================
# STEP 4: Define Dataset Paths
# ============================================================================
train_dir = 'NIAD-LL/Train'
val_dir   = 'NIAD-LL/Val'
test_dir  = 'NIAD-LL/Test'

In [ ]:
# ============================================================================
# STEP 5: Custom Dataset Class
# ============================================================================
class AnomalyDataset(Dataset):
    """
    Custom Dataset for loading normal and anomalous images.
    Expected structure:
        root_dir/
            ├── Normal/
            └── Anomalous/
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir   = root_dir
        self.transform  = transform
        self.images     = []
        self.labels     = []
        self.class_names = ['Normal', 'Anomalous']

        normal_dir = os.path.join(root_dir, 'Normal')
        if os.path.exists(normal_dir):
            for img_name in os.listdir(normal_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                    self.images.append(os.path.join(normal_dir, img_name))
                    self.labels.append(0)

        anomalous_dir = os.path.join(root_dir, 'Anomalous')
        if os.path.exists(anomalous_dir):
            for img_name in os.listdir(anomalous_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                    self.images.append(os.path.join(anomalous_dir, img_name))
                    self.labels.append(1)

        print(f"Loaded {len(self.images)} images from {root_dir}")
        print(f"Normal: {self.labels.count(0)}, Anomalous: {self.labels.count(1)}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            image = Image.new('RGB', (224, 224))
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
# ============================================================================
# STEP 6: Focal Loss (same as base model)
# ============================================================================
class FocalLoss(nn.Module):
    """
    Focal Loss for binary / multi-class classification.
    FL(pt) = -alpha_t * (1 - pt)^gamma * log(pt)

    Parameters
    ----------
    alpha : float  — balancing factor (weight for positive class).
                     Use 0.25 for the minority class when imbalanced.
    gamma : float  — focusing parameter.  0 = cross-entropy, 2 is standard.
    reduction : str — 'mean' | 'sum' | 'none'
    """
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # inputs : (B, C) raw logits
        # targets: (B,)  integer class indices
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt      = torch.exp(-ce_loss)                        # probability of correct class
        focal_w = self.alpha * (1.0 - pt) ** self.gamma
        loss    = focal_w * ce_loss

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

In [ ]:
# ============================================================================
# STEP 7: Data Transforms (same as base model)
# ============================================================================
print("\nDefining data transforms...")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# ============================================================================
# STEP 8: Load Datasets & DataLoaders (same as base model)
# ============================================================================
print("\nLoading datasets...")

train_dataset = AnomalyDataset(train_dir, transform=train_transform)
val_dataset   = AnomalyDataset(val_dir,   transform=val_test_transform)
test_dataset  = AnomalyDataset(test_dir,  transform=val_test_transform)

batch_size  = 32
num_workers = 2

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)

print(f"\nDataset Statistics:")
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

In [ ]:
# ============================================================================
# STEP 9: Model — CLIP ViT-B/32 (BASE visual encoder, no channel attention)
# ============================================================================
# open_clip is used so we can fine-tune the visual tower with PyTorch optimiser.
# The text encoder is discarded; only the visual backbone is retained.

import open_clip

# File-name constants
BEST_MODEL_PATH = 'best_clip_vitb32_model.pth'
TRAIN_FIG_PATH  = 'clip_vitb32_training_history.png'
CM_FIG_PATH     = 'clip_vitb32_confusion_matrix.png'
RESULTS_JSON_PATH = 'clip_vitb32_results.json'

print("\nInitialising CLIP ViT-B/32 visual encoder...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ── Wrapper: CLIP visual encoder + MLP classifier ─────────────────────────
class CLIPVisualClassifier(nn.Module):
    """
    CLIP ViT-B/32 visual encoder followed by an MLP classification head.
    No channel attention — pure baseline for comparison with the base model.
    Feature dimension from ViT-B/32 visual encoder = 512.
    """
    def __init__(self, num_classes=2):
        super().__init__()
        clip_model, _, _ = open_clip.create_model_and_transforms(
            'ViT-B-32', pretrained='openai')
        # Keep only the visual encoder
        self.visual = clip_model.visual

        feat_dim = 512   # ViT-B/32 output

        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # open_clip visual encoder returns (B, 512) after projection
        feat = self.visual(x)
        if isinstance(feat, (list, tuple)):
            feat = feat[0]
        # Flatten if needed
        if feat.dim() > 2:
            feat = feat.flatten(1)
        return self.classifier(feat)

model = CLIPVisualClassifier(num_classes=2).to(device)

# ── Selective freezing: freeze early transformer blocks, unfreeze last 4 ──
# ViT-B/32 has 12 transformer blocks (transformer.resblocks[0..11])
for param in model.parameters():
    param.requires_grad = False

for name, param in model.named_parameters():
    # Unfreeze last 4 transformer blocks + projection + classifier head
    unfreeze_keys = [
        'visual.transformer.resblocks.8',
        'visual.transformer.resblocks.9',
        'visual.transformer.resblocks.10',
        'visual.transformer.resblocks.11',
        'visual.proj',
        'visual.ln_post',
        'classifier'
    ]
    if any(name.startswith(k) for k in unfreeze_keys):
        param.requires_grad = True

trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_count     = sum(p.numel() for p in model.parameters())
print(f"Total parameters    : {total_count:,}")
print(f"Trainable parameters: {trainable_count:,}  ({100*trainable_count/total_count:.1f}%)")

# ── Note on transforms: CLIP was trained on its own normalisation.
#    Re-using ImageNet normalisation (same as base model) is intentional here
#    to keep the experimental variable isolated to the backbone only.
#    If you want CLIP-native normalisation, replace std/mean in the transforms
#    with open_clip.get_mean_std('ViT-B-32', pretrained='openai').

# ── Focal Loss (same as base model) ──────────────────────────────────────────
criterion = FocalLoss(alpha=0.25, gamma=2.0, reduction='mean')

# ── Optimizer with same LR strategy ──────────────────────────────────────────
backbone_params = [p for n, p in model.named_parameters()
                   if p.requires_grad and not n.startswith('classifier')]
head_params     = [p for n, p in model.named_parameters()
                   if p.requires_grad and n.startswith('classifier')]

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5, 'weight_decay': 1e-4},
    {'params': head_params,     'lr': 1e-4, 'weight_decay': 1e-4},
])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-7)

print(f"\nOptimizer param groups:")
print(f"  Backbone fine-tune: {len(backbone_params)} tensors  lr=1e-5")
print(f"  Head (classifier) : {len(head_params)} tensors  lr=1e-4")

In [ ]:
# ============================================================================
# STEP 10: Training Function
# ============================================================================
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total   = 0

    pbar = tqdm(dataloader, desc='Training')
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        _, predicted  = torch.max(outputs.data, 1)
        total        += labels.size(0)
        correct      += (predicted == labels).sum().item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100*correct/total:.2f}%'})

    return running_loss / len(dataloader), 100 * correct / total

In [ ]:
# ============================================================================
# STEP 11: Validation Function
# ============================================================================
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss    = 0.0
    correct         = 0
    total           = 0
    all_predictions = []
    all_labels      = []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validation')
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss    = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted  = torch.max(outputs.data, 1)
            total        += labels.size(0)
            correct      += (predicted == labels).sum().item()
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100*correct/total:.2f}%'})

    return running_loss / len(dataloader), 100 * correct / total, all_predictions, all_labels

In [ ]:
# ============================================================================
# STEP 12: Training Loop  —  CLIP ViT-B/32
# ============================================================================
num_epochs  = 50
best_val_acc = 0.0
train_losses, train_accs, val_losses, val_accs = [], [], [], []

print("\n" + "="*70)
print("Starting Training — CLIP ViT-B/32")
print("="*70)

for epoch in range(num_epochs):
    print(f'\nEpoch [{epoch+1}/{num_epochs}]')
    print('-' * 70)

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,  val_acc, _, _ = validate(model, val_loader, criterion, device)

    train_losses.append(train_loss);  train_accs.append(train_acc)
    val_losses.append(val_loss);      val_accs.append(val_acc)

    print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
    print(f'Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.2f}%')
    print(f'LR: {optimizer.param_groups[0]["lr"]:.2e}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch'              : epoch,
            'model_state_dict'   : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc'            : val_acc,
            'val_loss'           : val_loss,
        }, BEST_MODEL_PATH)
        print(f'✓ Best model saved! Val Acc: {val_acc:.2f}%')

    scheduler.step()

    if epoch > 20 and val_acc < best_val_acc - 5:
        print(f"\nEarly stopping triggered.  Best Val Acc: {best_val_acc:.2f}%")
        break

print("\n" + "="*70)
print("Training Completed!")
print("="*70)

In [ ]:
# ============================================================================
# STEP 13: Plot Training History  —  CLIP ViT-B/32
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(train_losses, label='Train Loss', marker='o', markersize=3)
axes[0].plot(val_losses,   label='Val Loss',   marker='s', markersize=3)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss — CLIP ViT-B/32')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(train_accs, label='Train Acc', marker='o', markersize=3)
axes[1].plot(val_accs,   label='Val Acc',   marker='s', markersize=3)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy — CLIP ViT-B/32')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(TRAIN_FIG_PATH, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# STEP 14: Load Best Model & Evaluate on Test Set
# ============================================================================
print("\n" + "="*70)
print("Testing Best Model...")
print("="*70)

checkpoint = torch.load(BEST_MODEL_PATH)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1}")

test_loss, test_acc, test_predictions, test_labels_list = validate(model, test_loader, criterion, device)

print(f'\n{"="*70}')
print("TEST RESULTS")
print(f'{"="*70}')
print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc:.2f}%')

In [ ]:
# ============================================================================
# STEP 15: Classification Report
# ============================================================================
class_names = ['Normal', 'Anomalous']
print(classification_report(test_labels_list, test_predictions,
                             target_names=class_names, digits=4))

precision, recall, f1, support = precision_recall_fscore_support(
    test_labels_list, test_predictions, average=None)

for i, name in enumerate(class_names):
    print(f'{name:12s} - Precision: {precision[i]:.4f}, '
          f'Recall: {recall[i]:.4f}, F1: {f1[i]:.4f}, Support: {support[i]}')

p_avg, r_avg, f1_avg, _ = precision_recall_fscore_support(
    test_labels_list, test_predictions, average='macro')
print(f'\nMacro Average - Precision: {p_avg:.4f}, Recall: {r_avg:.4f}, F1: {f1_avg:.4f}')

In [ ]:
# ============================================================================
# STEP 16: Confusion Matrix
# ============================================================================
cm = confusion_matrix(test_labels_list, test_predictions)
print(cm)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title(f'Confusion Matrix - Test Accuracy: {test_acc:.2f}%')
plt.tight_layout()
plt.savefig(CM_FIG_PATH, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# STEP 17: Save Results
# ============================================================================
import json

results = {
    'best_val_acc'           : best_val_acc,
    'test_acc'               : test_acc,
    'test_loss'              : test_loss,
    'classification_report'  : classification_report(test_labels_list, test_predictions,
                                                      target_names=class_names,
                                                      output_dict=True),
    'confusion_matrix'       : cm.tolist(),
    'train_losses'           : train_losses,
    'train_accs'             : train_accs,
    'val_losses'             : val_losses,
    'val_accs'               : val_accs,
}
with open(RESULTS_JSON_PATH, 'w') as f:
    json.dump(results, f, indent=4)

print("\n" + "="*70)
print("All results saved successfully!")
print(f"  Best model  : {BEST_MODEL_PATH}")
print(f"  Training fig: {TRAIN_FIG_PATH}")
print(f"  CM fig      : {CM_FIG_PATH}")
print(f"  Results JSON: {RESULTS_JSON_PATH}")
print("="*70)

In [ ]:
# ============================================================================
# STEP 18: Inference Function for New Images
# ============================================================================
def predict_image(image_path, model, transform, device, class_names=['Normal', 'Anomalous']):
    model.eval()
    image        = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output        = model(image_tensor)
        probabilities = torch.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, 1)
    return class_names[predicted.item()], confidence.item() * 100

print("\n✓ Pipeline complete! Use predict_image() for single-image inference.")